# Bayes and Naive Bayes: home assignment




In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
np.set_printoptions(precision=5, suppress=True)
print("Ready! Class order in manual examples: Insulator, Metal.")

Ready! Class order in manual examples: Insulator, Metal.


## Problem 1. Conditional probability

A new teaching dataset has rows [perovskite, not perovskite] and columns [oxide, not oxide]. Calculate the joint probability and both conditional probabilities.



In [ ]:
counts = np.array([[4, 2], [3, 6]])
p_joint = counts[0, 0] / counts.sum()  
p_p_given_o = counts[0, 0] / counts[:, 0].sum()  
p_o_given_p = counts[0, 0] / counts[0, :].sum()  
if any(v is None for v in [p_joint, p_p_given_o, p_o_given_p]):
    print('Complete the three TODOs.')
else:
    assert np.allclose([p_joint, p_p_given_o, p_o_given_p], [4/15, 4/7, 4/6])
    print('Check passed:', p_joint, p_p_given_o, p_o_given_p)

Check passed: 0.26666666666666666 0.5714285714285714 0.6666666666666666


**Explain (1–2 sentences):** Why do the two conditional probabilities differ?

**Your answer:** Each one divides by a different group. P(perovskite | oxide) only looks at the 7 oxides, and P(oxide | perovskite) only looks at the 6 perovskites. The 4 materials that are both are the same in each case, but they get divided by 7 in one and by 6 in the other.

## Problem 2. One positive screening result

A stable material occurs with prior probability 0.10. A filter flags 80% of stable and 20% of unstable materials. Compute P(flag) and P(stable | flag).



In [3]:
prior, sensitivity, fpr = 0.10, 0.80, 0.20
p_flag = sensitivity * prior + fpr * (1 - prior)
posterior = sensitivity * prior / p_flag

if p_flag is None or posterior is None:
    print('Complete both TODOs.')
else:
    assert np.isclose(p_flag, 0.26)
    assert np.isclose(posterior*p_flag, sensitivity*prior)
    print('Check passed. Posterior:', posterior)

Check passed. Posterior: 0.30769230769230776


**Explain (1–2 sentences):** Why is the answer different from the sensitivity of 0.80?

**Your answer:** Most materials are unstable (90 out of 100), so even a small 20% false alarm rate gives a lot of wrong flags. Only about 31% of the flagged materials are really stable, because 0.80 only tells us how many stable materials get flagged, not how many flagged materials are stable.

## Problem 3. A second independent filter

Use the same rates as Problem 2. Write a reusable Bayes update function; apply it twice starting at 0.10. Assume the filters are independent conditional on the true class.


In [4]:
def update(prior, sensitivity, fpr):
    p_flag = sensitivity * prior + fpr * (1 - prior)
    return sensitivity * prior / p_flag

first = update(0.10, 0.80, 0.20)
second = update(first, 0.80, 0.20)

if first is None or second is None:
    print('Complete the function and second update.')
else:
    assert np.isclose(first, 4/13)
    assert np.isclose(second, 0.64)
    print('Check passed:', first, second)

Check passed: 0.30769230769230776 0.64


**Explain (1–2 sentences):** Would applying the same deterministic filter twice justify this update?

**Your answer:** No. If we run the exact same filter on the same material it just gives the same answer again, so it tells us nothing new. The second update is only fair when the second filter is a separate test that gives fresh evidence.

## Problem 4. Beta update

Start with Beta(3,7). Observe 4 stable candidates in 12 trials. Calculate the posterior parameters and mean.


In [5]:
a, b, k, n = 3, 7, 4, 12
a_post = a + k
b_post = b + (n - k)
post_mean = a_post / (a_post + b_post)

if any(v is None for v in [a_post, b_post, post_mean]):
    print('Complete all TODOs.')
else:
    assert a_post == 7 and b_post == 15
    assert np.isclose(post_mean, a_post/(a_post+b_post))
    print('Check passed:', a_post, b_post, post_mean)

Check passed: 7 15 0.3181818181818182


**Explain (1–2 sentences):** Compare the posterior mean with the prior mean and the observed success fraction.

**Your answer:** The posterior mean is about 0.32. It sits between the prior mean of 0.30 and the observed fraction of 4/12, which is about 0.33, so the data pulled our belief a little bit from the prior towards what we actually saw.

## Problem 5. Discrete classification
Classify a non-oxide with coordination number 6 and density below 5 g/cm³. Rows are [Insulator, Metal]; columns are [oxide, CN=6, low density]. Complement only the oxide column.


In [ ]:
priors = np.array([2/3, 1/3])
base = np.array([[0.8, 0.5, 0.6], [0.3, 0.3, 0.4]])
observed = base.copy()
observed[:, 0] = 1 - observed[:, 0]  
scores = priors * np.prod(observed, axis=1)
posterior = scores / scores.sum()

if any(v is None for v in [observed, scores, posterior]):
    print('Complete all TODOs.')
else:
    assert np.allclose(scores, [0.04, 0.028])
    assert np.allclose(posterior, scores/scores.sum())
    print('Check passed. Posterior [I,M]:', posterior)

Check passed. Posterior [I,M]: [0.58824 0.41176]


**Explain (1–2 sentences):** Which class wins, and why is the decision less decisive than for an oxide?

**Your answer:** The insulator wins, with about 59% against 41% for metal. For a non-oxide, the oxide clue points towards metal and fights against the other clues and the prior, while for an oxide everything points to insulator and the answer would be about 93%.

## Problem 6. Laplace smoothing
Of 20 metals, zero have a binary feature. Of 40 insulators, eight have it. Compute smoothed P(feature=yes | class) for each class.



In [7]:
p_yes_metal = (0 + 1) / (20 + 2)
p_yes_insulator = (8 + 1) / (40 + 2)

if p_yes_metal is None or p_yes_insulator is None:
    print('Complete both TODOs.')
else:
    assert np.allclose([p_yes_metal,p_yes_insulator], [1/22,9/42])
    print('Check passed:', p_yes_metal, p_yes_insulator)

Check passed: 0.045454545454545456 0.21428571428571427


**Explain (1–2 sentences):** Why does an observed count of zero not establish physical impossibility?

**Your answer:** A zero just means we did not see it in the 20 metals we happened to have, and a small sample can easily miss something that is rare. The feature could still show up in some other metal, so we keep a small chance for it instead of saying it can never happen.

## Problem 7. Gaussian density

Write the Gaussian density formula. Evaluate x=0.7 for a metal atomic-size descriptor with mean 0.6 Å and standard deviation 0.2 Å. Also calculate the z-score.



In [ ]:
def normal_density(x, mu, sigma):
    
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
z = (0.7 - 0.6) / 0.2
density = normal_density(0.7, 0.6, 0.2)

if z is None or density is None:
    print('Complete both TODOs.')
else:
    assert np.isclose(z, 0.5)
    assert np.isclose(density, 1.7603266338)
    print('Check passed:', z, density)

Check passed: 0.4999999999999999 1.7603266338214976


**Explain (1–2 sentences):** Why is this density allowed to exceed one?

**Your answer:** The density is the height of the curve, not a probability. The curve here is very narrow (sigma is only 0.2), so it has to be tall for the whole area under it to add up to one, and the area is what really counts as probability.

## Problem 8. Gaussian Naive Bayes

Classify x=[5.0,0.7,0.6]. Features are density, atomic size and electronegativity difference. Calculate the 2×3 density array, class scores, and normalized probabilities.



In [9]:
x = np.array([5.0, 0.7, 0.6])
means = np.array([[4, 1.3, 2.1], [4, 0.6, 0.4]])
stds = np.array([[2, 0.2, 0.4], [2, 0.2, 0.3]])
priors = np.array([2/3, 1/3])
heights = np.exp(-0.5 * ((x - means) / stds) ** 2) / (stds * np.sqrt(2 * np.pi))
scores = priors * np.prod(heights, axis=1)
posterior = scores / scores.sum()

if any(v is None for v in [heights, scores, posterior]):
    print('Complete all TODOs.')
else:
    assert np.shape(heights) == (2,3)
    reference = np.exp(-0.5*((x-means)/stds)**2)/(stds*np.sqrt(2*np.pi))
    assert np.allclose(heights, reference)
    assert np.allclose(scores, priors*np.prod(reference, axis=1))
    assert np.allclose(posterior, scores/scores.sum())
    print('Check passed. Posterior [I,M]:', posterior)

Check passed. Posterior [I,M]: [0.00002 0.99998]


**Explain (1–2 sentences):** Why does density provide no discrimination here? Should the predicted probability be treated as experimentally established confidence?

**Your answer:** Density has the same mean and spread for both classes, so it gives the same number for insulator and metal and cancels out when we compare them. The probability is only what our simple model says using the numbers we gave it, so it should not be treated as confidence proven by experiments, and real data would be needed to trust it.